In [1]:
import numpy as np
import pandas as pd
import io
import itertools
import matplotlib
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib import cm
from matplotlib import colorbar
from datetime import date
import re
# from tqdm import tqdm
import time
# from pybedtools import BedTool
import random
import math
import pickle
import seaborn as sns
from scipy.stats import percentileofscore, pearsonr, rankdata, fisher_exact, describe
from scipy.spatial.distance import jaccard, euclidean, cosine, pdist
# import functools
# import operator
import gzip
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import sys
import os
from functools import partial
from contextlib import closing
from multiprocessing import Pool
import re
import h5py
import tabix
import upsetplot
from wordcloud import WordCloud
from sklearn.cluster import MeanShift
from scipy.cluster import hierarchy
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

today = date.today()

In [3]:
def kde_wrapper(data, bw=0.1):
    fig = plt.figure()
    fig.patch.set_facecolor('white')
    
    sns.kdeplot(data, bw_adjust=bw)
    print(describe(data))
    return fig

**Functions**

In [95]:
# def combine_scores_multiple(DHS_scaled_vals, DHS_similar_regions, TF_scaled_vals, TF_similar_regions, epi_scaled_vals,
#                             epi_similar_regions, coex_scaled_vals, coex_similar_regions, cooccur_scaled_vals, cooccur_similar_regions,
#                             database_scaled_vals, database_similar_regions, exper_scaled_vals, exper_similar_regions,fusion_scaled_vals, fusion_similar_regions,
#                             homology_scaled_vals, homology_similar_regions, neigh_scaled_vals, neigh_similar_regions, textmin_scaled_vals, textmin_similar_regions):
#     #max ind for combined score array
#     max_ind = np.max([DHS_similar_regions, TF_similar_regions, epi_similar_regions, coex_similar_regions,
#                      cooccur_similar_regions, database_similar_regions, exper_similar_regions, fusion_similar_regions, homology_similar_regions,
#                      neigh_similar_regions, textmin_similar_regions])

#     #DHS scores
#     DHS_arr = create_arr_multiple(DHS_similar_regions, DHS_scaled_vals, max_ind)
#     #epilogos scores
#     epi_arr = create_arr_multiple(epi_similar_regions, epi_scaled_vals, max_ind)
#     #TF motif scores
#     TF_arr = create_arr_multiple(TF_similar_regions, TF_scaled_vals, max_ind)
#     #coexpression scores
#     coex_arr = create_arr_multiple(coex_similar_regions, coex_scaled_vals, max_ind)
#     #cooccurrence scores
#     cooccur_arr = create_arr_multiple(cooccur_similar_regions, cooccur_scaled_vals, max_ind)
#     #database scores
#     database_arr = create_arr_multiple(database_similar_regions, database_scaled_vals, max_ind)
#     #experiment scores
#     exper_arr = create_arr_multiple(exper_similar_regions, exper_scaled_vals, max_ind)
#     #fusion scores
#     fusion_arr = create_arr_multiple(fusion_similar_regions, fusion_scaled_vals, max_ind)
#     #homology scores
#     homology_arr = create_arr_multiple(homology_similar_regions, homology_scaled_vals, max_ind)
#     #neighborhood scores
#     neigh_arr = create_arr_multiple(neigh_similar_regions, neigh_scaled_vals, max_ind)
#     #text mining scores
#     textmin_arr = create_arr_multiple(textmin_similar_regions, textmin_scaled_vals, max_ind)

    
#     #fuse STRING scores first
#     string_arr = 1 - ((1 - coex_arr) * (1 - cooccur_arr) * (1 - database_arr) * (1 - exper_arr) *
#                       (1 - fusion_arr) * (1 - homology_arr) * (1 - neigh_arr) * (1 - textmin_arr))
    
    
#     #scoring all metrics
#     score_arr = 1 - ((1 - epi_arr) * (1 - DHS_arr) * (1 - TF_arr) * (1 - coex_arr) * (1 - cooccur_arr) * (1 - database_arr) * (1 - exper_arr) *
#                      (1 - fusion_arr) * (1 - homology_arr) * (1 - neigh_arr) * (1 - textmin_arr))

#     #labeling most contributing metrics for each region
#     contributing_metrics_arr = np.argmax(np.array((DHS_arr, TF_arr, epi_arr, coex_arr, cooccur_arr, database_arr, exper_arr, fusion_arr,
#                                                   homology_arr, neigh_arr, textmin_arr)), axis=0)

#     return (score_arr, contributing_metrics_arr, max_ind, DHS_arr, TF_arr, epi_arr, coex_arr, 
#             cooccur_arr, database_arr, exper_arr, fusion_arr, homology_arr, neigh_arr, textmin_arr, string_arr)

def combine_scores_multiple(DHS_scaled_vals, DHS_similar_regions, TF_scaled_vals, TF_similar_regions, epi_scaled_vals,
                            epi_similar_regions, coex_scaled_vals, coex_similar_regions, cooccur_scaled_vals, cooccur_similar_regions,
                            database_scaled_vals, database_similar_regions, exper_scaled_vals, exper_similar_regions,fusion_scaled_vals, fusion_similar_regions,
                            homology_scaled_vals, homology_similar_regions, neigh_scaled_vals, neigh_similar_regions, textmin_scaled_vals, textmin_similar_regions):
    #max ind for combined score array
    max_ind = np.max([DHS_similar_regions, TF_similar_regions, epi_similar_regions, coex_similar_regions,
                     cooccur_similar_regions, database_similar_regions, exper_similar_regions, fusion_similar_regions, homology_similar_regions,
                     neigh_similar_regions, textmin_similar_regions])

    #DHS scores
    DHS_arr = create_arr_multiple(DHS_similar_regions, DHS_scaled_vals, max_ind)
    #epilogos scores
    epi_arr = create_arr_multiple(epi_similar_regions, epi_scaled_vals, max_ind)
    #TF motif scores
    TF_arr = create_arr_multiple(TF_similar_regions, TF_scaled_vals, max_ind)
    #coexpression scores
    coex_arr = create_arr_multiple(coex_similar_regions, coex_scaled_vals, max_ind)
    #cooccurrence scores
    cooccur_arr = create_arr_multiple(cooccur_similar_regions, cooccur_scaled_vals, max_ind)
    #database scores
    database_arr = create_arr_multiple(database_similar_regions, database_scaled_vals, max_ind)
    #experiment scores
    exper_arr = create_arr_multiple(exper_similar_regions, exper_scaled_vals, max_ind)
    #fusion scores
    fusion_arr = create_arr_multiple(fusion_similar_regions, fusion_scaled_vals, max_ind)
    #homology scores
    homology_arr = create_arr_multiple(homology_similar_regions, homology_scaled_vals, max_ind)
    #neighborhood scores
    neigh_arr = create_arr_multiple(neigh_similar_regions, neigh_scaled_vals, max_ind)
    #text mining scores
    textmin_arr = create_arr_multiple(textmin_similar_regions, textmin_scaled_vals, max_ind)

    
    #fuse STRING scores first
    string_arr = 1 - ((1 - coex_arr) * (1 - cooccur_arr) * (1 - database_arr) * (1 - exper_arr) *
                      (1 - fusion_arr) * (1 - homology_arr) * (1 - neigh_arr) * (1 - textmin_arr))
    
    
    #scoring all metrics
    chromatin_arr = 1 - ((1 - epi_arr) * (1 - DHS_arr) * (1 - TF_arr))

    #labeling most contributing metrics for each region
    contributing_chromatin_metrics_arr = np.argmax(np.array((DHS_arr, TF_arr, epi_arr)), axis=0)
    contributing_string_metrics_arr = np.argmax(np.array((coex_arr, cooccur_arr, database_arr,
                                                          exper_arr, fusion_arr, homology_arr,
                                                          neigh_arr, textmin_arr)), axis=0)

    return (chromatin_arr, string_arr, contributing_chromatin_metrics_arr, contributing_string_metrics_arr, max_ind, DHS_arr, 
            TF_arr, epi_arr, coex_arr, cooccur_arr, database_arr, exper_arr, fusion_arr, homology_arr, 
            neigh_arr, textmin_arr)

def region_return_multiple(score_arr, roi):

    #top 100 region inds and scores
    rec_inds = np.argsort(-score_arr)[:, :100]
    top_region_inds = np.hstack((roi.reshape(-1, 1), rec_inds))
    # top_region_scores = np.append(1, score_arr[rec_inds])

    return top_region_inds

def roi_url(roi, top_regions):
    today = date.today()
    roi_url_name = 'region_comparison_recommended_regions_roi_'+str(roi)+'_scale_25kb_CT' + today.strftime("%Y%m%d") + '.bed'
    bed_path_name = os.path.join('/home/ctrader/public_html/', roi_url_name)
    np.savetxt(bed_path_name, top_regions, delimiter='\t', fmt='%s')
    link_path1 = 'https://epilogos.altius.org:1234/?application=viewer&sampleSet=vC&mode=single&genome=hg38&model=18&complexity=KL&group=all&chrLeft=chr1&chrRight=chr1&start=8000&stop=35000&roiURL=https%3A%2F%2Fencode%3Acollabor8%40resources.altius.org%2F~ctrader%2F' + roi_url_name
    link_path2 = 'https://index.altius.org:1234/?application=viewer&mode=pd&chrLeft=chr11&chrRight=chr11&start=47329800&stop=47356800&roiURL=https%3A%2F%2Fencode%3Acollabor8%40resources.altius.org%2F~ctrader%2F' + roi_url_name
    return link_path1, link_path2, bed_path_name

def load_hdf5_files_multiple(hdf5_path, roi):
    #take in a file path and load scaled scores and similar regions
    hdf5_file = h5py.File(hdf5_path, 'r')
    similar_regions = hdf5_file['top_1000_similar_regions'][roi]
    comp_ranks = hdf5_file['ranked_regions'][roi]
    sigmoid_scaled_vals = hdf5_file['top_1000_sigmoid_scaled_vals'][roi]
#     scaled_vals = hdf5_file['top_1000_scaled_vals'][roi]
#     comp_sigmoid = hdf5_file['sigmoid_ranks'][roi]
    hdf5_file.close()
    return sigmoid_scaled_vals, similar_regions, comp_ranks

def load_STRING_hdf5_files_multiple(hdf5_path, roi):
    #load in tracking genes
    hdf5_file = h5py.File(hdf5_path, 'r')
    ori_genes = np.char.decode(hdf5_file['top_1000_original_genes'][roi], 'utf-8')
    dest_genes = np.char.decode(hdf5_file['top_1000_destination_genes'][roi], 'utf-8')
    similar_regions = hdf5_file['top_1000_similar_regions'][roi]
    scaled_vals = hdf5_file['top_1000_scaled_vals'][roi]
#     STRING_vals = hdf5_file['top_1000_STRING_vals'][roi]
    hdf5_file.close()
    return scaled_vals, similar_regions, ori_genes, dest_genes

def create_arr_multiple(similar_regions, scaled_vals, max_ind):
    #empty array
    arr = np.zeros((len(similar_regions), max_ind + 1))
    
    #find the values to change
    cols = similar_regions
    rows = np.array([np.arange(len(similar_regions))]*len(similar_regions[0])).T
    
    #set values
    arr[rows, cols] = scaled_vals
    return arr

def load_hdf5_genes_multiple(ori_genes, dest_genes, similar_regions, max_ind):
    #establishing gene arrays
    ori_arr = np.where(np.zeros((len(similar_regions), max_ind + 1))==0, None, None)
    dest_arr = np.where(np.zeros((len(similar_regions), max_ind + 1))==0, None, None)

    #filling gene arrays
    #find the values to change
    cols = similar_regions
    rows = np.array([np.arange(len(similar_regions))]*len(similar_regions[0])).T
#     print(cols)
#     print(rows)
#     print(ori_genes.shape)

    #set values
    ori_arr[rows, cols] = ori_genes
    dest_arr[rows, cols] = dest_genes
    return np.array([ori_arr, dest_arr])

def get_comp_order(path):
    if path[-5:]=='.hdf5':
        hdf5_file = h5py.File(path, 'r')
        comp_order = np.char.decode(hdf5_file['comp_order'][:])
        hdf5_file.close()
    else:
        f = open(path, 'r')
        comp_order = np.array(f.readline().strip().split('\t'))[3:]
        f.close()
    return comp_order

def load_rank_path(hdf5_path, roi):
    hdf5_file = h5py.File(hdf5_path, 'r')
    rank_arr = hdf5_file['ranked_regions'][roi]
    max_ranks = hdf5_file['max_ranks'][:]
    hdf5_file.close()
    return rank_arr, max_ranks

**Data**

In [86]:
DHS_hdf5_path = '/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221104.DHSsigmoideuc.hdf5'
TF_hdf5_path = '/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221104.TFsigmoideuc.hdf5'
epilogos_hdf5_path = '/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221104.epilogossigmoideuc.hdf5'
coex_hdf5_path = '/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221031.coexpression_human_transferred_maxSTRINGsimilarity.hdf5'
cooccur_hdf5_path = '/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221031.cooccurenceSTRINGsimilarity.hdf5'
database_hdf5_path = '/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221031.database_human_transferred_maxSTRINGsimilarity.hdf5'
exper_hdf5_path = '/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221031.experiments_human_transferred_maxSTRINGsimilarity.hdf5'
fusion_hdf5_path = '/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221031.fusionSTRINGsimilarity.hdf5'
homology_hdf5_path = '/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221031.homologySTRINGsimilarity.hdf5'
neigh_hdf5_path = '/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221031.neighborhood_human_transferred_maxSTRINGsimilarity.hdf5'
textmin_hdf5_path = '/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221031.textmining_human_transferred_maxSTRINGsimilarity.hdf5'
STRING_ranks_hdf5_path = '/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221109.STRINGsimilarityrankedregions.hdf5'

DHS_mapped_path = '/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221104.DHSmapped.bed'
TF_mapped_path = '/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221104.TFmapped.bed'
epilogos_mapped_path = '/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221104.epilogosmapped.bed'

regions_path = '/home/ctrader/region_similarity/data/Altius_25kb_hg38_maxMean_CT20220815.sorted.c1t3.bed'
gz_file_path = '/home/ctrader/region_similarity/data/Altius_25kb_hg38_maxMean_CT20220815.sorted.c1t3.rownum4.bed.gz'

geneID2name_path = '/home/ctrader/region_similarity/data/gene_proteinID2genename_dict_CT20220712.pickle'

DHS_base_path='/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221104.DHScounts.hdf5'
TF_base_path='/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221104.TFcounts.hdf5'
epi_base_path = '/home/ctrader/region_similarity/data/rank_based/regions_25kb_CT20221104.epilogosmeanmax.hdf5'

In [5]:
#loading order of each component
DHS_component_order = get_comp_order(DHS_mapped_path)
TF_component_order = get_comp_order(TF_mapped_path)
epilogos_component_order = get_comp_order(epilogos_mapped_path)

In [6]:
pickle_in = open(geneID2name_path, 'rb')
geneID2name = pickle.load(pickle_in)

In [7]:
genomic_location = 'chr1:198636000-198661000'
# genomic_location = 'chr1:198636000-198636001'
gz_tb = tabix.open(gz_file_path)
# roi = int(list(gz_tb.querys(genomic_location))[0][3])
roi = np.array(list(gz_tb.querys(genomic_location)))[:, 3].astype(int)
print(roi)

[6434 6435]


In [8]:
roi = np.array([78852])

In [9]:
# f = open(regions_path, 'r')
# regions = f.readlines()
# f.close()

# regions[19733].strip().split('\t')

In [10]:
t = time.time()

###Load rank and ind arrays
#DHS counts
DHS_scaled_vals, DHS_similar_regions, DHS_roi_ranks = load_hdf5_files_multiple(DHS_hdf5_path, roi)
#TF motif counts
TF_scaled_vals, TF_similar_regions, TF_roi_ranks = load_hdf5_files_multiple(TF_hdf5_path, roi)
#epilogos score sum and max
epi_scaled_vals, epi_similar_regions, epi_roi_ranks = load_hdf5_files_multiple(epilogos_hdf5_path, roi)
#coexpression STRING scores
coex_scaled_vals, coex_similar_regions, coex_ori, coex_dest = load_STRING_hdf5_files_multiple(coex_hdf5_path, roi)
#cooccurence STRING scores
cooccur_scaled_vals, cooccur_similar_regions, cooccur_ori, cooccur_dest = load_STRING_hdf5_files_multiple(cooccur_hdf5_path, roi)
#database STRING scores
database_scaled_vals, database_similar_regions, database_ori, database_dest = load_STRING_hdf5_files_multiple(database_hdf5_path, roi)
#experiments STRING scores
exper_scaled_vals, exper_similar_regions, exper_ori, exper_dest = load_STRING_hdf5_files_multiple(exper_hdf5_path, roi)
#fusion STRING scores
fusion_scaled_vals, fusion_similar_regions, fusion_ori, fusion_dest = load_STRING_hdf5_files_multiple(fusion_hdf5_path, roi)
#homology STRING scores
homology_scaled_vals, homology_similar_regions, homology_ori, homology_dest = load_STRING_hdf5_files_multiple(homology_hdf5_path, roi)
#neighborhood STRING scores
neigh_scaled_vals, neigh_similar_regions, neigh_ori, neigh_dest = load_STRING_hdf5_files_multiple(neigh_hdf5_path, roi)
#textmining STRING scores
textmin_scaled_vals, textmin_similar_regions, textmin_ori, textmin_dest = load_STRING_hdf5_files_multiple(textmin_hdf5_path, roi)

###Combine scores 
(chromatin_arr, string_arr, 
 contributing_chromatin_metrics_arr, 
 contributing_string_metrics_arr, 
 max_ind, DHS_arr, TF_arr, 
 epi_arr, coex_arr, cooccur_arr, 
 database_arr, exper_arr, fusion_arr, 
 homology_arr, neigh_arr, textmin_arr) = combine_scores_multiple(DHS_scaled_vals, DHS_similar_regions, 
                                                                 TF_scaled_vals, TF_similar_regions, 
                                                                 epi_scaled_vals, epi_similar_regions, 
                                                                 coex_scaled_vals, coex_similar_regions, 
                                                                 cooccur_scaled_vals, cooccur_similar_regions,
                                                                 database_scaled_vals, database_similar_regions, 
                                                                 exper_scaled_vals, exper_similar_regions, 
                                                                 fusion_scaled_vals, fusion_similar_regions,
                                                                 homology_scaled_vals, homology_similar_regions, 
                                                                 neigh_scaled_vals, neigh_similar_regions, 
                                                                 textmin_scaled_vals, textmin_similar_regions)

#load in gene names for STRING scores
coex_genes = load_hdf5_genes_multiple(coex_ori, coex_dest, coex_similar_regions, max_ind)
cooccur_genes = load_hdf5_genes_multiple(cooccur_ori, cooccur_dest, cooccur_similar_regions, max_ind)
database_genes = load_hdf5_genes_multiple(database_ori, database_dest, database_similar_regions, max_ind)
exper_genes = load_hdf5_genes_multiple(exper_ori, exper_dest, exper_similar_regions, max_ind)
fusion_genes = load_hdf5_genes_multiple(fusion_ori, fusion_dest, fusion_similar_regions, max_ind)
homology_genes = load_hdf5_genes_multiple(homology_ori, homology_dest, homology_similar_regions, max_ind)
neigh_genes = load_hdf5_genes_multiple(neigh_ori, neigh_dest, neigh_similar_regions, max_ind)
textmin_genes = load_hdf5_genes_multiple(textmin_ori, textmin_dest, textmin_similar_regions, max_ind)

###Return regions
top_region_inds = region_return_multiple(chromatin_arr, roi)

print(time.time()-t)

2.8439836502075195


In [13]:
#values for top 100 regions
cols = top_region_inds[:, 1:]
rows = np.array([np.arange(len(top_region_inds[:, 1:]))]*len(top_region_inds[:, 1:][0])).T

DHS_arr_top_100 = DHS_arr[rows, cols]
TF_arr_top_100 = TF_arr[rows, cols]
epi_arr_top_100 = epi_arr[rows, cols]

string_arr_top_100 = np.array((coex_arr, cooccur_arr, database_arr, exper_arr, fusion_arr, homology_arr, 
                               neigh_arr, textmin_arr))[:, rows, cols]

string_comp_order = ['Co-expression', 'Co-occurence', 'Co-annotation', 
                     'Interactions in vitro', 'Fusion in Other Species', 'Common Ancestry', 
                     'being Nearby', 'Co-citation']
colors = np.array(['#a6cee3', '#1f78b4', '#b2df8a', '#33a02c', '#fb9a99', '#e31a1c', '#fdbf6f', '#ff7f00', 
                   '#cab2d6', '#6a3d9a', '#ffff99'])

**Regional Sentences**

In [81]:
roi = np.arange(98629)

In [140]:
#load rank arrays
DHS_rank_arr, DHS_max_ranks = load_rank_path(DHS_hdf5_path, roi)
TF_rank_arr, TF_max_ranks = load_rank_path(TF_hdf5_path, roi)
epi_rank_arr, epi_max_ranks = load_rank_path(epilogos_hdf5_path, roi)
STRING_rank_arr, STRING_max_ranks = load_rank_path(STRING_ranks_hdf5_path, roi)

#load comp orders
DHS_comp_order = get_comp_order(DHS_mapped_path)
TF_comp_order = get_comp_order(TF_mapped_path)
epi_comp_order = get_comp_order(epilogos_mapped_path)
STRING_comp_order = get_comp_order(STRING_ranks_hdf5_path)

In [145]:
epi_conversion_dict = {
    'EnhA1': 'active enhancer 1',
    'EnhA2': 'active enhancer 2',
    'EnhBiv': 'bivalent enhancer', 
    'EnhG1': 'genic enhancer 1',
    'EnhG2': 'genic enhancer 2', 
    'EnhWk': 'weak enhancer', 
    'Het': 'heterochromatin', 
    'Quies': 'quiescent',
    'ReprPC': 'repressed polycomb', 
    'ReprPCWk': 'weak repressed polycomb',
    'TssA': 'active TSS',
    'TssBiv': 'bivalent TSS',
    'TssFlnk': 'flanking TSS',
    'TssFlnkD': 'downstream flanking TSS',
    'TssFlnkU': 'upstream flanking TSS',
    'Tx': 'strong transcription',
    'TxWk': 'weak transcription',
    'ZNF_Rpts': 'ZNF genes + repeats'
}

DHS_conversion_dict = {
    'Cancer_epithelial': 'cancer/epithelial',
    'Cardiac': 'cardiac',
    'Digestive': 'digestive',
    'Lymphoid': 'lymphoid',
    'Musculoskeletal': 'musculoskeletal',
    'Myeloid_erythroid': 'myeloid/erythroid',
    'Neural': 'neural',
    'Organdevel_renal': 'organ development/renal',
    'Placental_trophoblast': 'placental/trophoblast',
    'Primitive_embryonic': 'primitive/embryonic',
    'Pulmonarydevel': 'pulmonary development',
    'Renal_cancer': 'renal/cancer',
    'StromalA': 'stromal A',
    'StromalB': 'stromal B',
    'Tissueinvariant': 'tissue invariant',
    'Vascular_endothelial': 'vascular/endothelial'
}

TF_conversion_dict = {}
for comp in TF_comp_order:
    TF_conversion_dict[comp] = comp
    
STRING_conversion_dict = {
    'Co-expression' : 'co-expression',
    'Co-occurrence' : 'co-evolution',
    'Database' : 'co-annotation',
    'Experiment' : 'in vitro',
    'Fusion' : 'non-human fused',
    'Homology' : 'common ancestry',
    'Neighborhood' : 'neighboring',
    'Textmining' : 'literature based'
}

In [146]:
rank_arrs = [DHS_rank_arr, TF_rank_arr, epi_rank_arr[:, :18], epi_rank_arr[:, 18:], STRING_rank_arr]
comp_arrs = [DHS_comp_order, TF_comp_order, epi_comp_order, epi_comp_order, STRING_comp_order]
max_rank_arrs = [DHS_max_ranks, TF_max_ranks, epi_max_ranks[:18], epi_max_ranks[18:], STRING_max_ranks]
metric_name_arrs = ['DHS components', 'TF motifs', 'chromatin state themes', 'chromatin state peaks', 'gene interassociations']
conversion_dicts = [DHS_conversion_dict, TF_conversion_dict, epi_conversion_dict, epi_conversion_dict, STRING_conversion_dict]

In [102]:
def regional_sentence(rank_arrs, comp_arrs, max_rank_arrs, metric_name_arrs, conversion_dicts, max_rank=100):
    
    num_metrics = len(rank_arrs)
    num_regions = len(rank_arrs[0])

    #establish sentences
    sentences = np.array([''] * (num_regions), dtype=object)
    sentence_start = 'This region ranks in the top '
    sentences[:] = sentence_start

    #ranks for each metric
    for (rank_arr, comp_arr, 
         max_ranks, metric_name, 
         conversion_dict, metric_i) in zip(rank_arrs, comp_arrs, 
                                           max_rank_arrs, 
                                           metric_name_arrs, 
                                           conversion_dicts,
                                           range(num_metrics)):
        #find percent of rank compared to max rank of each component
        adjusted_ranks = rank_arr / max_ranks
        #find the top ranking component
        rank_sort_order = np.argsort(-adjusted_ranks, axis=1)
        ranks_sorted = -np.sort(-adjusted_ranks, axis=1)
        top_rank_ind = rank_sort_order[:, 0]
        top_rank_val = ranks_sorted[:, 0]
        #find name of top component
        top_rank_comp = comp_arr[top_rank_ind]
        #convert to a readable version of name
        top_rank_comp = np.array(list(map(conversion_dict.get, top_rank_comp)))
        top_comp_arr = np.char.add(top_rank_comp, np.array([' ']*num_regions))
        #sentence transition with metric type
        if metric_i != (num_metrics - 1):
            metric_name = metric_name + ', '
        else:
            metric_name = metric_name + '.'
        metric_arr = np.array([metric_name] * num_regions)

        #int of rank percentage
        rank_percent = np.ceil((1 - top_rank_val) * 100).astype(int)
        #set top 0% ranks to top 1% ranks
        rank_percent = np.where(rank_percent==0, 1, rank_percent)
        #make percent readable
        rank_percent_str = np.char.add(rank_percent.astype(str), np.array(['% of ']*num_regions))

        #create section of sentence for metric
        sentence_section = np.char.add(np.char.add(rank_percent_str, top_comp_arr), metric_arr)
        #remove metric from regions where rank percentage is not small (good) enough
        sentence_section = np.where(rank_percent > max_rank, '', sentence_section)
        #append metric section to sentence
        sentences = np.char.add(sentences.astype(str), sentence_section)

    #if region does not have a metric with small (good) enough percentage, replace
    sentences = np.where(sentences==sentence_start, 
                         'This region does not rank in the top ' + str(max_rank) + '% for any metric.', 
                         sentences)
    return sentences

In [540]:
t = time.time()
sentences = regional_sentence(rank_arrs, comp_arrs, max_rank_arrs, metric_name_arrs, conversion_dicts, max_rank=20)
time.time() - t

6.4991819858551025

In [103]:
def regional_sentence_2(rank_arrs, comp_arrs, max_rank_arrs, metric_name_arrs, 
                        conversion_dicts, max_rank=100, multiple_rank_thresh=5, 
                        max_comps_per_metric=3):
    
    num_metrics = len(rank_arrs)
    num_regions = len(rank_arrs[0])

    #establish sentences
    sentences = np.array([''] * num_regions, dtype=object)
    sentence_start = 'This region ranks in the top '
    sentences[:] = sentence_start

    sentence_section_arr = []
    sentence_section_min_ranks = []

    backup_rank = np.ones(num_regions)*101
    backup_comp = np.array([''] * num_regions, dtype=object)
    backup_metric = np.array([''] * num_regions, dtype=object)

    #ranks for each metric
    for (rank_arr, comp_arr, 
         max_ranks, metric_name, 
         conversion_dict, metric_i) in zip(rank_arrs, comp_arrs, 
                                           max_rank_arrs, 
                                           metric_name_arrs, 
                                           conversion_dicts,
                                           range(num_metrics)):

        #find percent of rank compared to max rank of each component
        adjusted_ranks = rank_arr / max_ranks
        #find the top ranking component
        rank_sort_order = np.argsort(-adjusted_ranks, axis=1)
        ranks_sorted = -np.sort(-adjusted_ranks, axis=1)

        #int of rank percentage
        ranks_percent = np.ceil((1 - ranks_sorted) * 100).astype(int)
        #set top 0% ranks to top 1% ranks
        ranks_percent = np.where(ranks_percent==0, 1, ranks_percent)

        ###the ranks to include
        #include all values that rank in the top multiple_rank_thresh percent
        ranks_mask = ranks_percent <= multiple_rank_thresh
        #include top component if rank is in the top max_rank percent
        ranks_mask[:, 0] = ranks_percent[:, 0] <= max_rank
        #number of components to include for each region 
        num_comps_per_region = np.count_nonzero(ranks_mask, axis=1)

        #find name of top component
        comps_ranked = comp_arr[rank_sort_order]
        #convert to a readable version of name
        comps_ranked = np.array(list(map(conversion_dict.get, comps_ranked.flatten()))).reshape(comps_ranked.shape)

        #metric in sentence form
        metric_arr = np.char.add(' ', np.array([metric_name] * num_regions))

        #backup for regions
        better_backup_mask = ranks_percent[:, 0] < backup_rank
        backup_rank[better_backup_mask] = ranks_percent[:, 0][better_backup_mask]
        backup_comp[better_backup_mask] = comps_ranked[:, 0][better_backup_mask]
        backup_metric[better_backup_mask] = metric_arr[better_backup_mask]

        #initializing the section of the sentence for current metric
        sentence_section = np.array([''] * num_regions)

        metric_ranks_per_region = []
        #for each component in order of how they rank for each region, add to sentence section
        for comp in range(len(comp_arr))[:max_comps_per_metric]:
            if ((len(ranks_mask[:, comp][ranks_mask[:, comp]]) == 0) & (comp!=0)):
                break
            #mask for component
            comp_mask = ~ranks_mask[:, comp]
            #percentage
            comp_rank_percent = ranks_percent[:, comp].astype(str)
            comp_rank_percent[comp_mask] = ''

            #saving the rank values for each region
            metric_ranks_per_region.append(np.where(comp_mask, np.nan, ranks_percent[:, comp]))

            #filler between percent and component name
            per_of = np.array(['% of '] * num_regions)
            per_of[comp_mask] = ''
            #component name
            comp_name = comps_ranked[:, comp]
            comp_name[comp_mask] = ''

            #ending
            comp_ending = np.array([''] * num_regions, dtype=object)
            #for regions with more components than max_comps_per_metric with ranks better than multiple_rank_thresh
            if (comp + 2) == max_comps_per_metric: #if there is 1 component left
                comp_ending[num_comps_per_region > (comp + 2)] = ' and '
            elif (comp + 1) == max_comps_per_metric: #if there are no components left
                comp_ending[num_comps_per_region > (comp + 2)] = ''
            else:
                comp_ending[num_comps_per_region > (comp + 2)] = ', '

            #for regions with 1 more component left under the multiple_rank_thresh
            if (comp + 1) == max_comps_per_metric: #if there are no components left
                comp_ending[num_comps_per_region==(comp + 2)] = ''
            else:
                comp_ending[num_comps_per_region==(comp + 2)] = ' and '

            #append comp info to sentence section
            sentence_section = np.char.add(sentence_section, comp_rank_percent)
            sentence_section = np.char.add(sentence_section, per_of)
            sentence_section = np.char.add(sentence_section, comp_name)
            sentence_section = np.char.add(sentence_section, comp_ending)

        #add the metric to the sentence section
        metric_arr[~ranks_mask[:, 0]] = ''
        sentence_section = np.char.add(sentence_section, metric_arr)

        sentence_section_arr.append(sentence_section)

        #average ranks for each region
        metric_ranks_per_region = np.array(metric_ranks_per_region).T
        sentence_section_min_ranks.append(np.nanmin(metric_ranks_per_region, axis=1))

    sentence_section_arr = np.array(sentence_section_arr).T
    sentence_section_min_ranks = np.array(sentence_section_min_ranks).T
    sorted_sections = np.argsort(sentence_section_min_ranks, axis=1)

    #to know where to place '.', ',' and 'and'
    num_sections_with_ranks = np.count_nonzero(~np.isnan(sentence_section_min_ranks), axis=1)

    #add sections to sentences
    for section in range(num_metrics):
        sentences = np.char.add(sentences.astype(str), 
                                sentence_section_arr[np.arange(len(sorted_sections)), 
                                                     sorted_sections[:, section]])
        section_ending = np.array([', '] * num_regions, dtype=object)
        section_ending[num_sections_with_ranks==(section + 1)] = '.'
        section_ending[num_sections_with_ranks<(section + 1)] = ''
        section_ending[num_sections_with_ranks==(section + 2)] = ' and '
        sentences = np.char.add(sentences.astype(str), 
                                section_ending)

    #fill with backup ranks for regions that don't match filters for any metric
    need_backup_mask = sentences==sentence_start
    #rank
    sentences[need_backup_mask] = np.char.add(sentences[need_backup_mask].astype(str), 
                                              backup_rank[need_backup_mask].astype(int).astype(str))
    #fill between rank and component
    sentences[need_backup_mask] = np.char.add(sentences[need_backup_mask].astype(str), 
                                              np.array(['% of '] * num_regions)[need_backup_mask])
    #component
    sentences[need_backup_mask] = np.char.add(sentences[need_backup_mask].astype(str), 
                                              backup_comp[need_backup_mask])
    #metric
    sentences[need_backup_mask] = np.char.add(sentences[need_backup_mask].astype(str), 
                                              backup_metric[need_backup_mask])
    #.
    sentences[need_backup_mask] = np.char.add(sentences[need_backup_mask].astype(str), 
                                              np.array(['.'] * num_regions)[need_backup_mask])
    return sentences

In [147]:
t = time.time()
sentences = regional_sentence_2(rank_arrs, comp_arrs, max_rank_arrs, metric_name_arrs, 
                                conversion_dicts, max_rank=20, multiple_rank_thresh=5, 
                                max_comps_per_metric=3)
time.time() - t

/tmp/ipykernel_73516/2609883389.py:116: RuntimeWarning: All-NaN slice encountered
  sentence_section_min_ranks.append(np.nanmin(metric_ranks_per_region, axis=1))


37.39818358421326

In [134]:
np.argmax(STRING_rank_arr/STRING_max_ranks, axis=0)

array([73001, 18220,   694, 42927,   751, 18220, 11024, 55419])

In [154]:
sentences[55419]

'This region ranks in the top 1% of co-annotation, 1% of literature based and 1% of in vitro gene interassociations, 3% of HD_9, 3% of SCRT1 and 4% of SOX_3 TF motifs and 5% of neural DHS components.'

In [77]:
STRING_comps = ['Co-expression', 'Co-occurrence', 'Database', 'Experiment', 'Fusion', 'Homology', 
                'Neighborhood', 'Textmining']